In [8]:
import requests
import pandas as pd
import numpy as np
import joblib
import json
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta
load_dotenv()

True

In [3]:
#load the trained model and the features 
model = joblib.load('../models/rf_climate_risk_model.pkl')

with open('../models/feature_names.json', 'r') as f:
    features = json.load(f)

print("Model and features loaded successfully!")
print(f"Model features: {features}")

Model and features loaded successfully!
Model features: ['temp_avg', 'temp_max', 'temp_min', 'rainfall', 'humidity', 'solar_radiation', 'wind_speed', 'temp_range', 'month', 'rainfall_7day_avg']


In [4]:
#weather fetcher function 
def get_forecast_weather(lat, lon, api_key):
    """
    fetches a 5 day weather forecast for the given location using OpenWeatherMap API
    Returns a clean dataframe with one row per day
    """
    url  = "https://api.openweathermap.org/data/2.5/forecast"

    param = {
        "lat": lat,
        "lon": lon,
        "appid": api_key,
        "units": "metric"
    }

    response = requests.get(url, params=param)

    if response.status_code != 200:
        print(f"API Error: {response.status_code} - {response.text}")
        print(response.json())
        return None
    
    data = response.json()

    #extract daily data from 3 hour forecasts
    daily_data = {}

    for item in data["list"]:
        date = item["dt_txt"].split(" ")[0]

        if date not in daily_data:
            daily_data[date] = {
                "date": date,
                "temp_readings": [],
                "temp_max_readings": [],
                "temp_min_readings": [],
                "rainfall_readings": [],
                "humidity_readings": [],
                "wind_speed_readings": [],
                'solar_radiation': 15.0
            }
        #accumulate readings for the day
        daily_data[date]['temp_readings'].append(
            item['main']['temp']
        )
        daily_data[date]['temp_max_readings'].append(
            item['main']['temp_max']
        )
        daily_data[date]['temp_min_readings'].append(
            item['main']['temp_min']
        )
        daily_data[date]['humidity_readings'].append(
            item['main']['humidity']
        )
        daily_data[date]['wind_speed_readings'].append(
            item['wind']['speed']
        )

        #rainfall is optional in the API response
        if "rain" in item:
            daily_data[date]['rainfall_readings'].append(
                item['rain'].get('3h', 0)
            )
        else:
            daily_data[date]['rainfall_readings'].append(0)
        
        #aggregate daily averages
        rows = []
    for date, values in daily_data.items():
        rows.append({
            'date': pd.to_datetime(date),
            'temp_avg': np.mean(values['temp_readings']),
            'temp_max': np.max(values['temp_max_readings']),
            'temp_min': np.min(values['temp_min_readings']),
            'rainfall': np.sum(values['rainfall_readings']),
            'humidity': np.mean(values['humidity_readings']),
            'wind_speed': np.mean(values['wind_speed_readings']),
            'solar_radiation': values['solar_radiation']
        })
    
    forecast_df = pd.DataFrame(rows).sort_values('date').reset_index(drop=True)
    return forecast_df

In [5]:
#build the engineer features for the forecast data 
def prepare_forecast_features(forecast_df):
    """
    Add the same engineered features the model was trained on.
    """
    df = forecast_df.copy()
    
    # Temperature range
    df['temp_range'] = df['temp_max'] - df['temp_min']
    
    # Month
    df['month'] = df['date'].dt.month
    
    # 7 day rolling rainfall average
    df['rainfall_7day_avg'] = df['rainfall'].rolling(
        window=7, min_periods=1
    ).mean()
    
    # Ensure all required features are present
    for feature in features:
        if feature not in df.columns:
            df[feature] = 0
            print(f"Warning: {feature} not available, defaulting to 0")
    
    return df[features]

In [6]:
#build a function to forecast risk for the next 5 days
def forecast_climate_risk(lat, lon, api_key):
    """
    Fetch forecast weather and predict risk for each day.
    Returns a dataframe with dates and risk scores.
    """
    # Get forecast weather
    print("Fetching weather forecast...")
    forecast_df = get_forecast_weather(lat, lon, api_key)
    
    if forecast_df is None:
        return None
    
    print(f"Got {len(forecast_df)} days of forecast data")
    
    # Prepare features
    X_forecast = prepare_forecast_features(forecast_df)
    
    # Predict risk for each day
    risk_predictions = model.predict(X_forecast)
    risk_probabilities = model.predict_proba(X_forecast)
    
    # Build results dataframe
    results = forecast_df[['date', 'temp_max', 'temp_min', 
                            'rainfall', 'humidity']].copy()
    results['risk_label'] = risk_predictions
    results['risk_score'] = (risk_probabilities[:, 2] * 100).round(1)
    results['risk_category'] = results['risk_label'].map({
        0: 'Low Risk',
        1: 'Moderate Risk', 
        2: 'High Risk'
    })
    
    return results

In [7]:
#build the recommendation function based on the risk category
def get_harvest_recommendation(maturity_days, risk_forecast_df):
    """
    Combine GDD maturity date with climate risk forecast
    to generate optimal harvest window.
    
    maturity_days: days until crop is ready from today
    risk_forecast_df: output from forecast_climate_risk()
    """
    
    if risk_forecast_df is None:
        return "Unable to generate recommendation - forecast unavailable."
    
    recommendation = {
        'maturity_days': maturity_days,
        'harvest_window': None,
        'warning': None,
        'daily_forecast': []
    }
    
    # Build daily forecast summary
    for idx, row in risk_forecast_df.iterrows():
        day_number = idx + 1
        is_mature = day_number >= maturity_days
        
        recommendation['daily_forecast'].append({
            'day': day_number,
            'date': row['date'].strftime('%B %d, %Y'),
            'risk_category': row['risk_category'],
            'risk_score': row['risk_score'],
            'temp_max': round(row['temp_max'], 1),
            'rainfall': round(row['rainfall'], 1),
            'harvest_possible': is_mature
        })
    
    # Find optimal harvest window
    # Look for first window of consecutive low risk days after maturity
    window_start = None
    window_end = None
    
    for day in recommendation['daily_forecast']:
        if day['harvest_possible']:
            if day['risk_category'] == 'Low Risk':
                if window_start is None:
                    window_start = day['day']
                window_end = day['day']
            else:
                if window_start is not None:
                    break  # Found our window, stop here
    
    # Generate recommendation text
    if window_start and window_end:
        recommendation['harvest_window'] = (window_start, window_end)
        recommendation['message'] = (
            f"Your crop matures in {maturity_days} days. "
            f"Best harvest window: Day {window_start} to Day {window_end}. "
            f"Conditions look favourable during this period."
        )
        recommendation['pidgin_message'] = (
            f"Your crop don almost ready in {maturity_days} days. "
            f"Make you harvest between Day {window_start} and Day {window_end}. "
            f"Weather go be better that time."
        )
    elif maturity_days > len(risk_forecast_df):
        recommendation['message'] = (
            f"Your crop matures in {maturity_days} days — "
            f"beyond our current forecast window. "
            f"Check back closer to maturity date for harvest recommendations."
        )
        recommendation['pidgin_message'] = (
            f"Your crop no ready yet. E go ready in {maturity_days} days. "
            f"Come back check am when e near ready."
        )
    else:
        recommendation['warning'] = "High risk period ahead"
        recommendation['message'] = (
            f"Your crop matures in {maturity_days} days but "
            f"climate conditions look unfavourable in the forecast window. "
            f"Consider protective measures: mulching, early cover, "
            f"or consult your extension officer."
        )
        recommendation['pidgin_message'] = (
            f"Your crop go ready in {maturity_days} days but "
            f"weather no look good. Talk to your extension officer "
            f"and cover your farm well."
        )
    
    return recommendation

In [9]:
#test the pipeline
API_KEY = os.getenv("API_KEY")

# Kaduna coordinates
LAT = 10.5222
LON = 7.4383

# Step 1 - Get risk forecast
risk_forecast = forecast_climate_risk(LAT, LON, API_KEY)

if risk_forecast is not None:
    print("14-Day Climate Risk Forecast for Kaduna:")
    print("=" * 55)
    print(risk_forecast[['date', 'temp_max', 'rainfall', 
                          'risk_category', 'risk_score']].to_string())
    
    # Step 2 - Get harvest recommendation
    # Assuming maize planted 60 days ago - matures in 31 more days
    maturity_days = 31
    
    recommendation = get_harvest_recommendation(
        maturity_days, risk_forecast
    )
    
    print("\n" + "=" * 55)
    print("HARVEST RECOMMENDATION")
    print("=" * 55)
    print(f"\nEnglish: {recommendation['message']}")
    print(f"\nPidgin: {recommendation['pidgin_message']}")
    
    print("\nDay by Day Forecast:")
    for day in recommendation['daily_forecast']:
        status = "✓ HARVEST POSSIBLE" if day['harvest_possible'] else "  Not ready yet"
        print(f"Day {day['day']:2d} | {day['date']} | "
              f"{day['risk_category']:15s} | "
              f"Score: {day['risk_score']:5.1f} | {status}")

Fetching weather forecast...
Got 6 days of forecast data
14-Day Climate Risk Forecast for Kaduna:
        date  temp_max  rainfall risk_category  risk_score
0 2026-04-27     37.63      0.67     High Risk        99.9
1 2026-04-28     39.09      0.00     High Risk        99.0
2 2026-04-29     38.14      0.00     High Risk        99.0
3 2026-04-30     38.08      0.00     High Risk        99.0
4 2026-05-01     39.29      0.00     High Risk        99.8
5 2026-05-02     27.35      0.00      Low Risk        15.6

HARVEST RECOMMENDATION

English: Your crop matures in 31 days — beyond our current forecast window. Check back closer to maturity date for harvest recommendations.

Pidgin: Your crop no ready yet. E go ready in 31 days. Come back check am when e near ready.

Day by Day Forecast:
Day  1 | April 27, 2026 | High Risk       | Score:  99.9 |   Not ready yet
Day  2 | April 28, 2026 | High Risk       | Score:  99.0 |   Not ready yet
Day  3 | April 29, 2026 | High Risk       | Score:  99.0 |

In [10]:
#Test with crop that matures in 4 days to trigger the harvest window logic
maturity_days = 4

recommendation = get_harvest_recommendation(
    maturity_days, risk_forecast
)

print("HARVEST RECOMMENDATION - SHORT MATURITY TEST")
print("=" * 55)
print(f"\nEnglish: {recommendation['message']}")
print(f"\nPidgin: {recommendation['pidgin_message']}")

print("\nDay by Day Forecast:")
for day in recommendation['daily_forecast']:
    status = "✓ HARVEST POSSIBLE" if day['harvest_possible'] else "  Not ready yet"
    print(f"Day {day['day']:2d} | {day['date']} | "
          f"{day['risk_category']:15s} | "
          f"Score: {day['risk_score']:5.1f} | {status}")

HARVEST RECOMMENDATION - SHORT MATURITY TEST

English: Your crop matures in 4 days. Best harvest window: Day 6 to Day 6. Conditions look favourable during this period.

Pidgin: Your crop don almost ready in 4 days. Make you harvest between Day 6 and Day 6. Weather go be better that time.

Day by Day Forecast:
Day  1 | April 27, 2026 | High Risk       | Score:  99.9 |   Not ready yet
Day  2 | April 28, 2026 | High Risk       | Score:  99.0 |   Not ready yet
Day  3 | April 29, 2026 | High Risk       | Score:  99.0 |   Not ready yet
Day  4 | April 30, 2026 | High Risk       | Score:  99.0 | ✓ HARVEST POSSIBLE
Day  5 | May 01, 2026 | High Risk       | Score:  99.8 | ✓ HARVEST POSSIBLE
Day  6 | May 02, 2026 | Low Risk        | Score:  15.6 | ✓ HARVEST POSSIBLE
